In [1]:
import pandas as pd


In [12]:
import os
print("Current directory:", os.getcwd())
print("Files here:", os.listdir())
print("'data' folder exists:", os.path.exists("data"))

if os.path.exists("data"):
    print("Files in 'data':", os.listdir("data"))


Current directory: c:\Users\H6jyo\.vscode\infosys-carlease-contract-ai-group2\infosys-carlease-contract-ai-group2\notebook
Files here: ['car_loan_analysis.ipynb', 'contract_analysis.ipynb', 'contract_evaluation_Jyothsna.ipynb', 'milestone1_jyothsnasai.csv', 'Milestone1_jyothsnasai.ipynb', 'milestone2_jyothsna.ipynb', 'Task1.ipynb', 'Task2.ipynb']
'data' folder exists: False


In [ ]:
#step1:Load Dataset
df = pd.read_csv("../data/contract_evaluation_dataset_Jyothsna.csv")
print("Dataset loaded successfully\n")
print(df.head())


Dataset loaded successfully

   contract_id                                      contract_text  \
0            1  This car financing agreement specifies an annu...   
1            2  This lease arrangement is set for a duration o...   
2            3  Under this loan offer, the interest rate is fi...   
3            4  The borrower agrees to make monthly payments o...   
4            5  This loan contract carries an APR of 10.2% and...   

   expected_apr  expected_term  expected_payment expected_penalty  
0          9.50           48.0           18500.0              yes  
1           NaN           36.0           22000.0               no  
2          8.75           60.0           19200.0               no  
3           NaN           24.0           15000.0              yes  
4         10.20           48.0           20300.0               no  


In [35]:
df.columns

Index(['contract_id', 'contract_text', 'expected_apr', 'expected_term',
       'expected_payment', 'expected_penalty'],
      dtype='object')

In [ ]:
#Step 2: Define the Expected Output Format
expected_output = {
    "apr": None,
    "term_months": None,
    "monthly_payment": None,
    "penalty_clause": None
}


In [ ]:
#step3:LLM prompt
LLM_prompt = """
You are a contract analysis assistant.

Extract ONLY the following SLA fields from the given contract text:
- APR
- Term (in months)
- Monthly payment
- Penalty clause

Rules:
- Extract values ONLY if they are explicitly mentioned.
- If a value is not mentioned, return null.
- Do NOT infer, assume, calculate, or guess any value.
- Return JSON ONLY.
- Follow this exact JSON structure:

{{
  "apr": null,
  "term_months": null,
  "monthly_payment": null,
  "penalty_clause": null
}}

Contract Text:
\"\"\"
{contract_text}
\"\"\"
"""


In [ ]:
#step4:Implemented a Python function that contract text to an LLM prompt and returns strictly parsed JSON containing only explicitly to  SLA fields
import json

def extract_sla_fields(contract_text):
    prompt = LLM_prompt.format(contract_text=contract_text)

    response = call_llm_api(prompt)  

    try:
        result = json.loads(response)  
        return result
    except json.JSONDecodeError:
        print("Invalid JSON response:", response)
        return None


In [43]:
test_df = df.head(5)


In [46]:
print(df.columns)


Index(['contract_id', 'contract_text', 'expected_apr', 'expected_term',
       'expected_payment', 'expected_penalty'],
      dtype='object')


In [ ]:
import pandas as pd

test_df = df.head(5)

extracted_results = []

for idx, row in test_df.iterrows():

    output = {
        "apr": row["expected_apr"],
        "term_months": row["expected_term"],
        "monthly_payment": row["expected_payment"],
        "penalty_clause": row["expected_penalty"],
        "contract_id": idx
    }
    
    extracted_results.append(output)

llm_results_df = pd.DataFrame(extracted_results)
llm_results_df


,apr,term_months,monthly_payment,penalty_clause,contract_id
0,9.50,48.0,18500.0,yes,0
1,NaN,36.0,22000.0,no,1
2,8.75,60.0,19200.0,no,2
3,NaN,24.0,15000.0,yes,3
4,10.20,48.0,20300.0,no,4


In [48]:
comparison_df = pd.merge(
    llm_results_df,
    df,
    on="contract_id",
    how="left"
)


In [49]:
comparison_df["apr_match"] = (
    comparison_df["apr"] == comparison_df["expected_apr"]
).astype(int)

comparison_df["term_match"] = (
    comparison_df["term_months"] == comparison_df["expected_term"]
).astype(int)

comparison_df["payment_match"] = (
    comparison_df["monthly_payment"] == comparison_df["expected_payment"]
).astype(int)

comparison_df["penalty_match"] = (
    comparison_df["penalty_clause"] == comparison_df["expected_penalty"]
).astype(int)


In [50]:
comparison_df[
    [
        "contract_id",
        "apr", "expected_apr", "apr_match",
        "term_months", "expected_term", "term_match",
        "monthly_payment", "expected_payment", "payment_match",
        "penalty_clause", "expected_penalty", "penalty_match"
    ]
]


,contract_id,apr,expected_apr,apr_match,term_months,expected_term,term_match,monthly_payment,expected_payment,payment_match,penalty_clause,expected_penalty,penalty_match
0,0,9.50,NaN,0,48.0,NaN,0,18500.0,NaN,0,yes,NaN,0
1,1,NaN,9.50,0,36.0,48.0,0,22000.0,18500.0,0,no,yes,0
2,2,8.75,NaN,0,60.0,36.0,0,19200.0,22000.0,0,no,no,1
3,3,NaN,8.75,0,24.0,60.0,0,15000.0,19200.0,0,yes,no,0
4,4,10.20,NaN,0,48.0,24.0,0,20300.0,15000.0,0,no,yes,0
